In [1]:
%pip install -q transformers datasets evaluate rouge_score sentencepiece accelerate nltk sumy pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import numpy as np
import pandas as pd
import nltk
import evaluate

from datasets import load_dataset
from transformers import (
    PegasusTokenizer,
    PegasusForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

nltk.download("punkt")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU. Training will be much slower.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "xlsum-train.parquet",
        "validation": "xlsum-validation.parquet",
        "test": "xlsum-test.parquet",
    }
)

train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
val_dataset = dataset["validation"].shuffle(seed=42).select(range(500))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(dataset)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 38242
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 4780
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 4780
    })
})
Train samples: 3000
Validation samples: 500
Test samples: 500


In [4]:
print("Columns:", train_dataset.column_names)

sample = train_dataset[0]

print("\nARTICLE SAMPLE:")
print(sample["text"][:1000])

print("\nREFERENCE SUMMARY:")
print(sample["summary"])

Columns: ['id', 'url', 'title', 'summary', 'text']

ARTICLE SAMPLE:
Bulan Januari lalu, militan ISIS membebaskan anggota sekte Yazidi yang mereka tahan. Diketahui bahwa mereka yang dilepaskan adalah orang-orang yang sudah berusia lanjut atau dalam keadaan sakit. Komandan pasukan Kurdi, Westa Rasul, menyatakan beberapa di antara yang dilepaskan adalah 'wanita dan anak-anak'. Puluhan ribu penganut sekte Yazidi dipaksa untuk mengungsi ketika ISIS menyerbu desa-desa mereka pada bulan Agustus tahun lalu. Ribuan dari mereka mengungsi hingga ke gunung-gunung di Irak utara. Kelompok militan ini mengutuk keimanan sekte Yazidi dan membunuh ratusan anggota komunitas ini. Perserikatan Bangsa-bangsa menyatakan kemungkinan bahwa ISIS melakukan pemusnahan massal atau genosida terhadap sekte Yazidi. Ribuan lain anggota sekte ini ditangkap dan ditahan. Kebanyakan dari mereka adalah perempuan yang dilaporkan dijual untuk dijadikan budak seks. Bulan Januari lalu, militan ISIS juga membebaskan pengikut se

In [5]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

def textrank_summarize(text, sentence_count=3):
    try:
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        summarizer = TextRankSummarizer()
        summary = summarizer(parser.document, sentence_count)
        return " ".join(str(sentence) for sentence in summary)
    except Exception:
        return ""

In [6]:
rouge = evaluate.load("rouge")

baseline_predictions = []
baseline_references = []

for item in test_dataset:
    pred = textrank_summarize(item["text"])
    baseline_predictions.append(pred)
    baseline_references.append(item["summary"])

baseline_scores = rouge.compute(
    predictions=baseline_predictions,
    references=baseline_references
)

print("TextRank Baseline ROUGE Scores:")
baseline_scores

TextRank Baseline ROUGE Scores:


{'rouge1': np.float64(0.0),
 'rouge2': np.float64(0.0),
 'rougeL': np.float64(0.0),
 'rougeLsum': np.float64(0.0)}

In [7]:
model_name = "google/pegasus-xsum"

tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Device:", device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda


In [8]:
def pegasus_summarize(text, max_input_length=512, max_output_length=64):
    inputs = tokenizer(
        text,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_output_length,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

base_predictions = []
base_references = []

# Use only 20 samples for zero-shot test to save time.
for item in test_dataset.select(range(20)):
    pred = pegasus_summarize(item["text"])
    base_predictions.append(pred)
    base_references.append(item["summary"])

base_scores = rouge.compute(
    predictions=base_predictions,
    references=base_references
)

print("Base PEGASUS Zero-Shot ROUGE Scores:")
base_scores

Base PEGASUS Zero-Shot ROUGE Scores:


{'rouge1': np.float64(0.1531090526328373),
 'rouge2': np.float64(0.0389517517748942),
 'rougeL': np.float64(0.13358000118475755),
 'rougeLsum': np.float64(0.13306631582783268)}

In [9]:
max_input_length = 512
max_target_length = 64

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

print(tokenized_train)
print(tokenized_val)
print(tokenized_test)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {key: round(value * 100, 4) for key, value in result.items()}

In [11]:
# Gradient checkpointing reduces GPU memory usage.
model.gradient_checkpointing_enable()

# Disable cache because it conflicts with gradient checkpointing during training.
model.config.use_cache = False

training_args = Seq2SeqTrainingArguments(
    output_dir="./pegasus-xlsum-indonesia-3000",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [12]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,16.498151,3.589957,20.485500,6.303000,16.632300,16.629700
2,15.494004,3.450477,20.319000,6.124300,16.583600,16.550800
3,15.031056,3.414421,20.862800,6.498400,16.837000,16.820600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1125, training_loss=16.252402804904513, metrics={'train_runtime': 1694.3591, 'train_samples_per_second': 5.312, 'train_steps_per_second': 0.664, 'total_flos': 1.2714766476066816e+16, 'train_loss': 16.252402804904513, 'epoch': 3.0})

In [13]:
test_results = trainer.evaluate(tokenized_test)

print("Fine-Tuned PEGASUS Test Results:")
test_results

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
15.031056,3.407911,3,21.122300,7.034500,17.083700,17.091000


Fine-Tuned PEGASUS Test Results:


{'eval_loss': 3.4079108238220215,
 'eval_rouge1': 21.1223,
 'eval_rouge2': 7.0345,
 'eval_rougeL': 17.0837,
 'eval_rougeLsum': 17.091}

In [14]:
def fine_tuned_pegasus_summarize(text):
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [15]:
for i in range(5):
    article = test_dataset[i]["text"]
    reference = test_dataset[i]["summary"]

    textrank_summary = textrank_summarize(article)
    pegasus_summary = fine_tuned_pegasus_summarize(article)

    print("=" * 100)
    print(f"SAMPLE {i + 1}")

    print("\nARTICLE:")
    print(article[:1000])

    print("\nREFERENCE SUMMARY:")
    print(reference)

    print("\nTEXTRANK BASELINE:")
    print(textrank_summary)

    print("\nFINE-TUNED PEGASUS:")
    print(pegasus_summary)

SAMPLE 1

ARTICLE:
Pandemi Covid-19 telah meningkatkan minat berinvestasi emas. Kenaikan harga itu salah satu penyebab utamanya karena permainan para pedagang emas dan permintaan yang tinggi, tapi di balik itu, muncul pertanyaan tentang berapa sisa pasokan logam mulia itu di bumi dan kapan akan habis. Emas menjadi buruan masyarakat karena dapat dijadikan sebagai investasi, simbol status ekonomi, dan komponen utama produk elektronik. Tapi jumlah emas di dunia terbatas, dan pada akhirnya akan datang satu saat ketika tidak ada lagi emas yang tersisa untuk ditambang. Puncak produksi emas Beberapa ahli berbicara tentang apakah dunia telah mencapai puncak produksi emas - diukur dengan jumlah terbanyak emas yang pernah ditambang dalam satu tahun - atau tidak. Hal itu ditunjukan dengan mulai menurunnya tren produksi emas dunia. Contohnya, pada tahun 2019, produksi tambang emas dunia turun 1% menjadi 3.531 ton dibandingkan tahun 2018, menurut Dewan Emas Dunia. Penurunan produksi tahunan ini yan

In [16]:
model.save_pretrained("./final-pegasus-xlsum-indonesia-1000")
tokenizer.save_pretrained("./final-pegasus-xlsum-indonesia-1000")

print("Model saved to: ./final-pegasus-xlsum-indonesia-1000")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./final-pegasus-xlsum-indonesia-1000


In [17]:
model.save_pretrained("./final-pegasus-xlsum-indonesia-3000")
tokenizer.save_pretrained("./final-pegasus-xlsum-indonesia-3000")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final-pegasus-xlsum-indonesia-3000\\tokenizer_config.json',
 './final-pegasus-xlsum-indonesia-3000\\tokenizer.json')

In [18]:
comparison = pd.DataFrame({
    "Model": [
        "TextRank Extractive Baseline",
        "Base PEGASUS Zero-Shot",
        "Fine-Tuned PEGASUS"
    ],
    "Dataset Used": [
        "100 test samples",
        "20 test samples",
        "800 train / 100 validation / 100 test"
    ],
    "ROUGE Results": [
        str(baseline_scores),
        str(base_scores),
        str(test_results)
    ]
})

comparison

,Model,Dataset Used,ROUGE Results
0,TextRank Extractive Baseline,100 test samples,"{'rouge1': np.float64(0.0), 'rouge2': np.float..."
1,Base PEGASUS Zero-Shot,20 test samples,"{'rouge1': np.float64(0.1531090526328373), 'ro..."
2,Fine-Tuned PEGASUS,800 train / 100 validation / 100 test,"{'eval_loss': 3.4079108238220215, 'eval_rouge1..."


In [19]:
custom_article = """
Masukkan artikel berita bahasa Indonesia di sini.
"""

if custom_article.strip():
    print(fine_tuned_pegasus_summarize(custom_article))
else:
    print("Please insert an Indonesian article first.")

Masukkan artikel beritasa Indonesia di sini.


In [20]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))
print(training_args.per_device_train_batch_size)
print(training_args.gradient_accumulation_steps)
print(training_args.num_train_epochs)

3000
500
500
2
4
3
